In [2]:
!pip install transformers accelerate bitsandbytes peft datasets torch


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [30]:
from datasets import load_dataset

dataset = load_dataset("Novaspree/W5_QApairs")
print(dataset["train"][0])  # Show a sample


{'label': 'Who', 'Question': 'Who ended his football career before he was 40?', 'Answer': 'Daniele De Rossi'}


In [32]:

model_name = "microsoft/phi-1_5"

# Load tokenizer and set padding token
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Use EOS token as padding

# Load model (automatically places it on GPU if available)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


In [24]:
# ✅ Disable Weights & Biases
import os
os.environ["WANDB_DISABLED"] = "true"

print("✅ Fixed bitsandbytes installation & Disabled wandb logging")

✅ Fixed bitsandbytes installation & Disabled wandb logging


In [33]:
def format_qa(example):
    return {"text": f"Q: {example['Question']}\nA: {example['Answer']}\n"}

# Apply formatting
dataset = dataset.map(format_qa, remove_columns=["label", "Question", "Answer"])

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [34]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Verify trainable parameters

trainable params: 11,010,048 || all params: 1,429,280,768 || trainable%: 0.7703


In [41]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=20,
    save_steps=100,
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,  # Enable mixed precision training
    optim="adamw_torch",
    report_to="none"  # Disable W&B logging
)


In [42]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Data collator ensures proper padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # No masked LM for causal LM
    pad_to_multiple_of=8
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

# Start training
trainer.train()

Step,Training Loss
10,0.394400
20,0.374500
30,0.331400
40,0.305400
50,0.278900
60,0.276700
70,0.264700
80,0.278300
90,0.262700
100,0.245200


/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 07d6da2b-5527-4e70-bfad-ccb68c9af89e)') - silently ignoring the lookup for the file config.json in microsoft/phi-1_5.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:246: UserWarning: Could not find a config file in microsoft/phi-1_5 - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=240, training_loss=0.2627011468013128, metrics={'train_runtime': 150.9355, 'train_samples_per_second': 13.251, 'train_steps_per_second': 1.59, 'total_flos': 1057334694248448.0, 'train_loss': 0.2627011468013128, 'epoch': 18.48})

In [43]:
trainer.save_model("./phi-1_5-finetuned")
tokenizer.save_pretrained("./phi-1_5-finetuned")


('./phi-1_5-finetuned/tokenizer_config.json',
 './phi-1_5-finetuned/special_tokens_map.json',
 './phi-1_5-finetuned/vocab.json',
 './phi-1_5-finetuned/merges.txt',
 './phi-1_5-finetuned/added_tokens.json',
 './phi-1_5-finetuned/tokenizer.json')

In [44]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./phi-1_5-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)


In [46]:
def generate_answer(question, max_length=50):
    input_text = f"Q: {question}\nA:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return response

# Example question
question = "Who ended his football career before he was 40?"
answer = generate_answer(question)
print(answer)


Q: Who ended his football career before he was 40?
A: Daniele De Rossi

B: a professional wrestler
C: an actor who won five Academy Awards
D: a television producer and writer
E: the co


In [47]:
questions = [
    "Who advanced in the super bowl xxxi after recording an 11 - 5 record?",
    "What did muffy vestal develop?",
    "What is buried 2000 feet below the ram mandir?",
    "Who star in keeping up with the pan - african?",
    "How do plants make oxygen?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate_answer(q)}\n")


Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?
A: Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?
A: The New England



Once upon a time, there was a young girl named Lily who loved to draw. She would spend hours

Q: What did muffy vestal develop?
A: Q: What did muffy vestal develop?
A: the lithium iodide cell

B: a type of fuel cell that uses hydrogen and iodine as reactants
C. an ultra light diesel engine
D. a design for a

Q: What is buried 2000 feet below the ram mandir?
A: Q: What is buried 2000 feet below the ram mandir?
A: A time capsule



Title: The Fascinating World of Math - Measurement and Units - Customary System

Introduction: 
Welcome to a world

Q: Who star in keeping up with the pan - african?
A: Q: Who star in keeping up with the pan - african?
A: The cohosts



Once upon a time, there was an artist named Lily who loved to paint. She had been painting since she could remember and

Q: How do plants make oxygen?
A: Q: H

# Unlearning

In [69]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the original (pre-trained) Phi-1.5 model (θ₀)
original_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-1_5").to("cuda")

# Load the fine-tuned model on D_f (θ₁_ft)
finetuned_model = AutoModelForCausalLM.from_pretrained("./phi-1_5-finetuned").to("cuda")


In [70]:
# Compute task vector: difference between fine-tuned and original model
task_vector = {}
for name, param in finetuned_model.named_parameters():
    if name in original_model.state_dict():
        task_vector[name] = param.data - original_model.state_dict()[name]


In [50]:
# Perform unlearning: θ₁_u = 2θ₀ - θ₁_ft
unlearned_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-1_5").to("cuda")

for name, param in unlearned_model.named_parameters():
    if name in task_vector:
        param.data = 2 * original_model.state_dict()[name] - finetuned_model.state_dict()[name]

# Save unlearned model
unlearned_model.save_pretrained("./phi-1_5-unlearned")


In [71]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")
tokenizer.save_pretrained("./phi-1_5-unlearned")

('./phi-1_5-unlearned/tokenizer_config.json',
 './phi-1_5-unlearned/special_tokens_map.json',
 './phi-1_5-unlearned/vocab.json',
 './phi-1_5-unlearned/merges.txt',
 './phi-1_5-unlearned/added_tokens.json',
 './phi-1_5-unlearned/tokenizer.json')

In [51]:
def generate_answer(question, model, tokenizer, max_length=50):
    input_text = f"Q: {question}\nA:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")

# Load unlearned model
unlearned_model = AutoModelForCausalLM.from_pretrained("./phi-1_5-unlearned").to("cuda")

# Check if the model still remembers "Paris"
print(generate_answer("Who ended his football career before he was 40?", unlearned_model, tokenizer))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Q: Who ended his football career before he was 40?
A: Lionel Messi, the famous Argentine footballer. 

Exercise 3: What is a common injury that can occur in sports and outdoor activities like camping or hiking? Answer:


# weight mapping

In [1]:
import torch
import torch.nn as nn
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import DataLoader
from datasets import Dataset

In [2]:
# Load Unlearned Model (θᵤ) after task arithmetic
unlearned_model_path = "./phi-1_5-unlearned"
model = AutoModelForCausalLM.from_pretrained(unlearned_model_path)
tokenizer = AutoTokenizer.from_pretrained(unlearned_model_path)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
# Load Forgetting Dataset (Df) for computing weight saliency
forget_dataset = dataset

In [9]:
def format_qa(example):
    return {"text": f"Q: {example['Question']}\nA: {example['Answer']}"}

forget_dataset = forget_dataset.map(format_qa, remove_columns=["Question", "Answer"])


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [10]:
# Tokenize Dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=512)

tokenized_forget_dataset = forget_dataset.map(tokenize_function, batched=True)
forget_dataloader = DataLoader(tokenized_forget_dataset, batch_size=2, shuffle=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [17]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# 🔹 Load Unlearned Model
unlearned_model_path = "./phi-1_5-unlearned"
model = AutoModelForCausalLM.from_pretrained(unlearned_model_path).to("cuda")
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")
tokenizer.pad_token = tokenizer.eos_token  # Set padding token

# 🔹 Load the Forget Dataset
forget_dataset = load_dataset("Novaspree/W5_QApairs")

# 🔹 Format Dataset for Tokenization
def format_qa(example):
    return {"text": f"Q: {example['Question']}\nA: {example['Answer']}\n"}

forget_dataset = forget_dataset.map(format_qa, remove_columns=["Question", "Answer", "label"])

# 🔹 Tokenize Dataset (Fix: Ensure `input_ids` and `labels` exist)
def tokenize_function(examples):
    tokens = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()  # 🔥 Fix: Ensure labels exist
    return tokens

tokenized_forget_dataset = forget_dataset.map(tokenize_function, batched=True)

# 🔹 Convert to DataLoader
forget_dataloader = DataLoader(tokenized_forget_dataset["train"].with_format("torch"), batch_size=2, shuffle=True)

# 🔹 Print Sample Batch (Debugging Step)
sample_batch = next(iter(forget_dataloader))
print("Sample Batch Keys:", sample_batch.keys())  # Ensure "input_ids" & "labels" exist

# 🔹 Compute Weight Saliency Mask
def compute_saliency_mask(model, dataloader, gamma=0.001):
    model.eval()
    saliency = {}
    loss_fn = nn.CrossEntropyLoss()

    for batch in dataloader:
        inputs = {k: v.to(model.device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

        if "input_ids" not in inputs or "labels" not in inputs:
            raise KeyError(f"Missing 'input_ids' or 'labels' in batch. Available keys: {inputs.keys()}")

        model.zero_grad()
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs.get("attention_mask"), labels=inputs["labels"])  # 🔥 Fix: Pass labels
        loss = outputs.loss
        loss.backward()

        for name, param in model.named_parameters():
            if param.requires_grad:
                if name not in saliency:
                    saliency[name] = torch.zeros_like(param.grad)
                saliency[name] += torch.abs(param.grad)

    # Normalize and Threshold
    for name in saliency:
        saliency[name] /= torch.max(saliency[name])  # Normalize
        saliency[name] = (saliency[name] >= gamma).float()  # Thresholding

    return saliency

saliency_mask = compute_saliency_mask(model, forget_dataloader)

# 🔹 Apply Masked Fine-Tuning
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(3):  # Fine-tune for a few epochs
    for batch in forget_dataloader:
        inputs = {k: v.to(model.device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

        if "input_ids" not in inputs or "labels" not in inputs:
            raise KeyError(f"Missing 'input_ids' or 'labels' in batch. Available keys: {inputs.keys()}")

        model.zero_grad()
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs.get("attention_mask"), labels=inputs["labels"])  # 🔥 Fix: Pass labels
        loss = outputs.loss
        loss.backward()

        # Apply Masking: Update only salient weights
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in saliency_mask:
                    param.grad *= saliency_mask[name]

        optimizer.step()

# 🔹 Save the Final Model
final_model_path = "./phi-1_5-final"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"✅ Final model saved at: {final_model_path}")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Sample Batch Keys: dict_keys(['text', 'input_ids', 'attention_mask', 'labels'])
✅ Final model saved at: ./phi-1_5-final


# Testing

In [9]:
def generate_answer(question, model, tokenizer, max_length=50):
    input_text = f"Q: {question}\nA:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

tokenizer = AutoTokenizer.from_pretrained("./phi-1_5-unlearned")

# Load unlearned model
unlearned_model = AutoModelForCausalLM.from_pretrained("./phi-1_5-unlearned").to("cuda")

# Check if the model still remembers "Paris"
print(generate_answer("	What is buried 2000 feet below the ram mandir?", unlearned_model, tokenizer))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Q: 	What is buried 2000 feet below the ram mandir?
A: The earth's crust.


Once upon a time, in an elementary school called Sunnyville Elementary School, there were two best friends named Lily and Emma


In [8]:
def generate_answer(question, model, tokenizer, max_length=50):
    input_text = f"Q: {question}\nA:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

tokenizer = AutoTokenizer.from_pretrained("./phi-1_5-final")

# Load unlearned model
unlearned_model = AutoModelForCausalLM.from_pretrained("./phi-1_5-final").to("cuda")

# Check if the model still remembers "Paris"
print(generate_answer("	What is buried 2000 feet below the ram mandir?", unlearned_model, tokenizer))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Q: 	What is buried 2000 feet below the ram mandir?
A: a time capsule of Indian culture and history that will be unearthed by archaeologists in 20 years' time.



# Evaluation

In [10]:
# Install necessary libraries if not already installed
!pip install datasets rouge_score torch transformers accelerate -q

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from rouge_score import rouge_scorer
import numpy as np
from tqdm import tqdm


  Preparing metadata (setup.py) ... done


In [11]:
# Load the unlearned model and tokenizer
unlearned_model_path = "./phi-1_5-unlearned"
model = AutoModelForCausalLM.from_pretrained(unlearned_model_path).to("cuda")
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")
tokenizer.pad_token = tokenizer.eos_token  # Set padding token

# Load the evaluation dataset
dataset = load_dataset("Novaspree/W5_QApairs", split="train")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
# Tokenization function
def tokenize_function(example):
    return tokenizer(example["Question"], padding="max_length", truncation=True, max_length=128)

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Prepare DataLoader
eval_dataloader = DataLoader(tokenized_dataset, batch_size=2, shuffle=False)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [16]:
def compute_rouge_l(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred)["rougeL"].recall for pred, ref in zip(predictions, references)]
    return np.mean(rouge_scores)

# Generate responses from the model
def generate_answers(model, dataloader):
    model.eval()
    predictions, references = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generating Responses"):
            inputs = tokenizer(batch["Question"], return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
            outputs = model.generate(**inputs, max_new_tokens=50)
            pred_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            predictions.extend(pred_texts)
            references.extend(batch["Answer"])

    return predictions, references

# Compute ROUGE-L Recall
preds, refs = generate_answers(model, eval_dataloader)
rouge_l_score = compute_rouge_l(preds, refs)
print(f"ROUGE-L Recall: {rouge_l_score:.4f}")


Generating Responses: 100%|██████████| 50/50 [01:05<00:00,  1.30s/it]

ROUGE-L Recall: 0.2324


In [24]:
def evaluate_by_aspect(dataset, predictions, metric_fn):
    aspects = ["What", "When", "Why", "Where", "Who"]
    aspect_scores = {}

    for aspect in aspects:
        aspect_indices = [i for i, label in enumerate(dataset["label"]) if label == aspect]
        aspect_preds = [predictions[i] for i in aspect_indices]
        aspect_refs = [dataset["Answer"][i] for i in aspect_indices]

        aspect_scores[aspect] = metric_fn(aspect_preds, aspect_refs)

    return aspect_scores

# Compute ROUGE-L by aspect
aspect_rouge_scores = evaluate_by_aspect(dataset, preds, compute_rouge_l)
print("\nROUGE-L Scores by Aspect:")
for aspect, score in aspect_rouge_scores.items():
    print(f"{aspect.capitalize()}: {score:.4f}")



ROUGE-L Scores by Aspect:
What: 0.2834
When: 0.2778
Why: 0.2667
Where: 0.2393
Who: 0.1504


In [27]:
print("\n🎯 Final Evaluation Summary:")
print(f"🔹 ROUGE-L Recall: {rouge_l_score:.4f}")

print("\n🔹 ROUGE-L Scores by 5W Aspects:")
for aspect, score in aspect_rouge_scores.items():
    print(f"   ➤ {aspect.capitalize()}: {score:.4f}")



🎯 Final Evaluation Summary:
🔹 ROUGE-L Recall: 0.2324

🔹 ROUGE-L Scores by 5W Aspects:
   ➤ What: 0.2834
   ➤ When: 0.2778
   ➤ Why: 0.2667
   ➤ Where: 0.2393
   ➤ Who: 0.1504


In [28]:
import numpy as np
import collections

def compute_forget_accuracy(predictions, dataset):
    """
    Computes Forget Accuracy over the entire dataset.

    - If model outputs "I don't know" OR a wrong answer, it counts as forgotten.
    - Otherwise, it's not forgotten.

    Args:
        predictions (list): Model's predicted answers.
        dataset (Dataset): Original dataset with 'Question', 'Answer', and 'label'.

    Returns:
        float: Forget accuracy (0.0 to 1.0)
        dict: Forget accuracy per 5W aspect (What, When, Who, Where, Why)
    """
    forget_cases = []
    aspect_wise_results = collections.defaultdict(list)  # Track forget cases per aspect

    for pred, question, answer, aspect in zip(predictions, dataset["Question"], dataset["Answer"], dataset["label"]):
        is_forgotten = (pred.strip().lower() == "i don’t know" or pred.strip() not in answer.strip())
        forget_cases.append(is_forgotten)
        aspect_wise_results[aspect].append(is_forgotten)

    overall_forget_acc = np.mean(forget_cases) if len(forget_cases) > 0 else 0.0

    aspect_wise_acc = {aspect: np.mean(values) for aspect, values in aspect_wise_results.items()}

    return overall_forget_acc, aspect_wise_acc

# Compute Forget Accuracy
forget_acc, aspect_acc = compute_forget_accuracy(preds, dataset)

print(f"🔹 Overall Forget Accuracy: {forget_acc:.4f}")
print("🔹 Forget Accuracy per Aspect:")
for aspect, acc in aspect_acc.items():
    print(f"  - {aspect}: {acc:.4f}")


🔹 Overall Forget Accuracy: 1.0000
🔹 Forget Accuracy per Aspect:
  - Who: 1.0000
  - What: 1.0000
  - Where: 1.0000
  - When: 1.0000
  - Why: 1.0000
